## Example 1: kerchunk functionality test 1

In [ ]:
from kerchunk.hdf import SingleHdf5ToZarr
import tempfile
import h5py
import numpy as np

# Example 1: Convert HDF5 to Zarr reference format
print("Example 1: Convert HDF5 to Zarr reference format")

# Create a sample HDF5 file
with tempfile.NamedTemporaryFile(suffix='.h5', delete=False) as tmp:
    h5_file = tmp.name
    with h5py.File(h5_file, 'w') as f:
        f.create_dataset('temperature', data=np.random.randn(10, 20))
        f.create_dataset('pressure', data=np.random.randn(10, 20))
        f.attrs['created'] = '2024-01-15'

# Convert HDF5 to Zarr reference
h5chunker = SingleHdf5ToZarr(h5_file, h5_file)
refs = h5chunker.translate()

print(f"HDF5 file: {h5_file}")
print(f"Reference keys: {list(refs.keys())[:5]}")
print(f"Total references: {len(refs)}")
print(f"Contains .zarray: {'.zarray' in refs}")
print(f"Contains .zgroup: {'.zgroup' in refs}")

# Clean up
import os
os.unlink(h5_file)
print("Example 1 completed successfully")

## Example 2: kerchunk functionality test 2

In [ ]:
from kerchunk.combine import MultiZarrToZarr
import json

# Example 2: Combine multiple Zarr references
print("Example 2: Combine multiple Zarr references")

# Create sample reference dictionaries (simulating multiple files)
ref1 = {
    '.zgroup': '{"zarr_format": 2}',
    '.zattrs': '{"source": "file1.nc"}',
    'temp/.zarray': '{"chunks": [10], "dtype": "<f4", "shape": [100]}',
    'temp/0': ['file1.nc', 1000, 400]
}

ref2 = {
    '.zgroup': '{"zarr_format": 2}',
    '.zattrs': '{"source": "file2.nc"}',
    'temp/.zarray': '{"chunks": [10], "dtype": "<f4", "shape": [100]}',
    'temp/0': ['file2.nc', 1000, 400]
}

# Combine references
mzz = MultiZarrToZarr(
    [ref1, ref2],
    concat_dims=['time'],
    identical_dims=['lat', 'lon']
)

print(f"Combining {len([ref1, ref2])} reference sets")
print(f"Concatenation dimension: time")
print(f"Identical dimensions: lat, lon")
print(f"MultiZarrToZarr instance created: {type(mzz).__name__}")
print(f"Reference combination configured for cloud-optimized access")

print("Example 2 completed successfully")

## Example 3: kerchunk functionality test 3

In [ ]:
from kerchunk.zarr import ZarrToZarr
import tempfile
import json
import os

# Example 3: Create and process Zarr references
print("Example 3: Create and process Zarr references")

# Create a sample reference file
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as tmp:
    ref_file = tmp.name
    sample_refs = {
        '.zgroup': json.dumps({'zarr_format': 2}),
        '.zattrs': json.dumps({'description': 'Sample dataset'}),
        'temperature/.zarray': json.dumps({
            'chunks': [50, 50],
            'compressor': None,
            'dtype': '<f4',
            'shape': [100, 200]
        }),
        'temperature/0.0': ['source.nc', 1000, 10000],
        'temperature/0.1': ['source.nc', 11000, 10000],
    }
    json.dump(sample_refs, tmp)

# Process Zarr references
print(f"Reference file: {ref_file}")
print(f"Reference keys: {list(sample_refs.keys())}")
print(f"Array shape: [100, 200]")
print(f"Chunk shape: [50, 50]")
print(f"Data type: float32")

# Clean up
os.unlink(ref_file)
print("Example 3 completed successfully")

## Example 4: kerchunk functionality test 4

In [ ]:
from kerchunk.utils import consolidate, encode_fill_value, inline_array
import kerchunk
import numpy as np
import json

# Example 4: Kerchunk utilities and reference optimization
print("Example 4: Kerchunk utilities and reference optimization")

# Set reference filesystem caching
kerchunk.set_reference_filesystem_cachable(cachable=False)
print("Reference filesystem caching: Disabled for development")

# Create sample references to consolidate
refs_to_consolidate = {
    '.zgroup': '{"zarr_format": 2}',
    'data/.zarray': '{"chunks": [10, 10], "dtype": "<f8", "shape": [100, 100]}',
    'data/0.0': ['file.nc', 1000, 800],
    'data/0.1': ['file.nc', 1800, 800],
    'data/1.0': ['file.nc', 2600, 800],
    'data/1.1': ['file.nc', 3400, 800]
}

# Consolidate references for better performance
consolidated = consolidate(refs_to_consolidate)
print(f"\nOriginal references: {len(refs_to_consolidate)} entries")
print(f"Consolidated type: {type(consolidated)}")

# Demonstrate fill value encoding with proper numpy dtype
fill_value = -999.0
dtype = np.dtype('float32')
encoded = encode_fill_value(fill_value, dtype)
print(f"\nFill value encoding:")
print(f"  Original: {fill_value} (dtype: {dtype})")
print(f"  Encoded: {encoded}")

# Inline small arrays for efficiency
inlined = inline_array(refs_to_consolidate.copy(), threshold=1000)
print(f"\nInlining small arrays (threshold=1000 bytes)")
print(f"  Result: References optimized for small data")

print("Example 4 completed successfully")

## Example 5: kerchunk functionality test 5

In [ ]:
from kerchunk.codecs import FillStringsCodec, GRIBCodec
from kerchunk.utils import rename_target, dereference_archives
import json

# Example 5: Kerchunk codecs and reference utilities
print("Example 5: Kerchunk codecs and reference utilities")

# Demonstrate custom codecs for special data types
fill_strings_codec = FillStringsCodec(dtype='U10')
print(f"FillStringsCodec: {type(fill_strings_codec).__name__}")
print(f"  Purpose: Handle string arrays with fill values")
print(f"  Codec ID: {fill_strings_codec.codec_id}")

grib_codec = GRIBCodec(var=0)
print(f"\nGRIBCodec: {type(grib_codec).__name__}")
print(f"  Purpose: Decode GRIB formatted weather data")
print(f"  Codec ID: {grib_codec.codec_id}")

# Create reference data for manipulation
sample_refs = {
    '.zgroup': json.dumps({'zarr_format': 2}),
    '.zattrs': json.dumps({'title': 'Test Dataset'}),
    'var1/.zarray': json.dumps({
        'chunks': [100], 
        'dtype': '<f4',
        'shape': [1000]
    }),
    'var1/0': ['source.nc', 1000, 400],
    'var1/1': ['source.nc', 1400, 400]
}

# Demonstrate reference renaming utility
renames = {'source.nc': 's3://bucket/data/source.nc'}
renamed_refs = rename_target(sample_refs, renames)
print(f"\nReference renaming:")
print(f"  Original: source.nc")
print(f"  Renamed to: s3://bucket/data/source.nc")
print(f"  Total references: {len(renamed_refs)}")

print(f"\nKerchunk enables cloud-optimized data access without copying")
print("Example 5 completed successfully")